# OCR Motor Karşılaştırması — Gerçek Derlem, A100

`glm-ocr:latest`, `deepseek-ocr:latest` ve `frob/unlimited-ocr:q8_0` motorlarını,
23 gerçek taranmış resmî yazışma belgesi (52 sayfa) üzerinde, projenin gerçek
üretim çıkarım sınıflarıyla (`FallbackDocumentExtractor`, `OllamaVisionExtractor`)
karşılaştırır. Skorlama mantığı `scripts/evaluate_ocr_real.py`'de yaşıyor ve
burada **yeniden yazılmıyor** — script bu notebook'tan olduğu gibi çağrılıyor,
tek doğruluk kaynağı üretim koduyla aynı kalsın diye.

**İki alan grubu ayrı puanlanır**: başlık bölgesi (Sayı/Tarih/Konu/Muhatap/
Gönderen — `header_repair` zaten her motoru burada eşitliyor) ve imza bölgesi
(İmza sahibi/unvanı — motorların asıl ayrıştığı yer, çünkü ıslak imza basılı
ismi bazen tamamen yok ediyor).

**Tahmini süre**: motor başına ~35-40 dk (52 sayfa × ham geçiş + zincir geçişi),
4 motor + Tesseract tabanı için toplam **~2.5-3 saat**. Sonuçlar motor bazında
birikimli JSON'a yazılıyor — oturum yarıda kesilirse tamamlanan motorlar kaybolmaz.

**Çalıştırmadan önce**: Colab menüsünden *Çalışma zamanı → Çalışma zamanı türünü
değiştir → A100 GPU* seçili olmalı. Ayrıca sol paneldeki 🔑 (Secrets) sekmesine
`GITHUB_TOKEN` adıyla, repoyu klonlayabilen bir GitHub personal access token
eklenmeli (Colab bunu notebook dosyasına hiç yazmaz, yalnızca oturum belleğinde
tutar).

## 1. GPU doğrulama

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()
print("Yukarıda 'A100' görmüyorsanız: Çalışma zamanı > Çalışma zamanı türünü değiştir > A100 GPU, sonra bu hücreyi tekrar çalıştırın.")

## 2. Repo klonu

`GITHUB_TOKEN` Colab Secrets'tan okunur, hiçbir hücreye yazılmaz. Dal parametrik —
bu OCR-skorlama düzeltmesinin bulunduğu `fix/ocr-benchmark-signature-scoring`
dalını klonluyoruz (skorlama mantığındaki imza-alanı düzeltmesi bu dalda; henüz
`main`'e alınmadıysa buradan devam edin, alındıysa `main` olarak değiştirin).

In [ ]:
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO = "chyp3r/KACHOW-Teknofest-2026"
BRANCH = "fix/ocr-benchmark-signature-scoring"  # main'e alındıysa "main" yapın

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
!git clone --branch {BRANCH} --depth 1 {clone_url} repo 2>&1 | sed "s/{GITHUB_TOKEN}/***/g"
%cd repo
!git log -1 --oneline -- scripts/evaluate_ocr_real.py

## 3. Sistem paketleri

Backend imajının kendisiyle birebir aynı üç paket
(`deploy/docker/backend.Dockerfile`): Java, `OpenDataLoaderExtractor`'ın PDF
ayrıştırıcısı için zorunlu; Tesseract + Türkçe dil verisi, zincirin taban
motoru için. Biri eksik kalırsa o çıkarıcı sessizce devre dışı kalır ve zincir
üretimden farklı davranır — bu yüzden `apt-get`'in kendisi de doğrulanıyor.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y default-jre-headless tesseract-ocr tesseract-ocr-tur
!java -version
!tesseract --version | head -1
!tesseract --list-langs | grep -q tur && echo 'tur dil paketi OK' || echo 'HATA: tur dil paketi eksik'

## 4. Python bağımlılıkları

`backend/requirements.txt` olduğu gibi kurulur — skorlama mantığını burada
yeniden yazmıyoruz, gerçek üretim sınıflarını (`FallbackDocumentExtractor`,
`OllamaVisionExtractor`, `parse_labelled_fields`) import edip kullanıyoruz.
`torch` yok, birkaç dakika sürer.

In [ ]:
!pip install -q -r backend/requirements.txt nest_asyncio
import sys
sys.path.insert(0, "backend")

# Colab/Jupyter çekirdeği kendi asyncio event loop'unu çalışır durumda
# tutabiliyor; evaluate_ocr_real.py'nin main()/'ı ve bu notebook'un
# duman testi doğrudan asyncio.run() çağırıyor -- bu, zaten çalışan bir
# loop içinde "RuntimeError: this event loop is already running" ile
# patlar. nest_asyncio bunu iç içe geçmeye izin verecek şekilde yamalar;
# üretim kodunun (backend içindeki gerçek servis) hiçbir zaman girmediği
# bir durum, yalnızca bu notebook ortamına özgü.
import nest_asyncio
nest_asyncio.apply()

from app.infrastructure.extractors import FallbackDocumentExtractor  # noqa: F401 -- import doğrulaması
print("Backend paketleri OK")

## 5. Ollama kurulumu + modeller

`OLLAMA_BASE_URL` ayarına dokunmuyoruz: `settings.OLLAMA_BASE_URL` varsayılanı
zaten `http://localhost:11434` ve Colab'da Ollama tam orada çalışıyor (Docker'daki
`host.docker.internal` yönlendirmesi burada **yanlış** olurdu, o yüzden hiç
ayarlanmıyor).

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request, urllib.error

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Sabit bir sleep yerine gerçekten hazır olduğunu yokla -- serve'in port'a
# ne zaman bağlanacağı makineye göre değişir, sabit süre bazen erken
# (ilk model pull'u "connection refused" ile patlar) bazen gereksiz uzun olur.
for attempt in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        print(f"Ollama hazır ({attempt + 1}. denemede).")
        break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(1)
else:
    raise RuntimeError("Ollama 30 saniyede ayağa kalkmadı -- `!ollama serve` çıktısını kontrol edin.")

!ollama --version

In [ ]:
for model in ["glm-ocr:latest", "deepseek-ocr:latest", "frob/unlimited-ocr:q8_0"]:
    print(f"--- çekiliyor: {model} ---")
    !ollama pull {model}
!ollama list

## 6. Duman testi (kapı)

**Tam koşuma geçmeden önce bu hücre geçmeli.** Tek belge (CY-050 — ıslak imzanın
basılı ismi tamamen yok ettiği belge), tek motor (`glm-ocr`). Beklenen: imza
alanları puanlamaya dahil (`expected` alan sayısı **7**, 5 değil) ve log'da tam
sayfa yükseltmesinin gerçekten tetiklendiğini gösteren bir satır görünmeli
(`"Replaced [...]'s first page with a full-page vision transcription"`).
Bu iki şey doğrulanmadan 7. hücreyi çalıştırmayın — 3 saati bir kablolama
hatasını bulmak için harcamamak adına.

In [ ]:
import asyncio, importlib.util, logging
logging.basicConfig(level=logging.INFO)

spec = importlib.util.spec_from_file_location("ev", "scripts/evaluate_ocr_real.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

gt = ev._load_ground_truth()
docs = ev._load_documents(gt)
doc = [d for d in docs if d[0].startswith("CY-050")][0]
print("beklenen alanlar:", sorted(doc[2].keys()), "-- 7 olmalı (imza dahil)")
assert len(doc[2]) >= 5 and any(k.startswith("imza") for k in doc[2]), "İmza alanları skorlamada yok -- script eski sürüm mü?"

name, engine = ev._parse_engine_spec("ollama:glm-ocr:latest")
smoke = asyncio.run(ev._run_engine(name, engine, [doc]))
sig = smoke["totals"]["chain"]["signature"]
print(f"\nzincir imza sonucu: {sig['found']}/{sig['expected']} bulundu")
print("Yukarıdaki log'da 'full-page vision transcription' satırını gördüyseniz kapı geçti -- 7. hücreye geçin.")

## 7. Tam koşum

4 motor: Tesseract tabanı + 3 vision modeli. Sonuçlar motor bazında
`ocr_real_results.json`'a birikimli yazılır (`_save_results_file`, script'in
kendi içinde) — bir motor çökerse önceki motorların sonucu kaybolmaz, ve
`--report-only` mantığıyla ara durum her an okunabilir (aşağıdaki hücrede).

**~2.5-3 saat sürer.** Colab oturumu kopabilir; her motor bitiminde sonuç zaten
diskte, bu yüzden burada arada bir 8. hücreyi (ara tablo) çalıştırıp ilerlemeyi
kontrol edebilirsiniz.

In [ ]:
import sys
sys.argv = [
    "evaluate_ocr_real.py",
    "--engine", "tesseract",
    "--engine", "ollama:glm-ocr:latest",
    "--engine", "ollama:deepseek-ocr:latest",
    "--engine", "ollama:frob/unlimited-ocr:q8_0",
]
ev.main()

## 8. Ara/son durum tablosu

Tam koşum bitmeden de, o ana kadar tamamlanan motorların tablosunu görmek için
bu hücre bağımsız çalıştırılabilir (`--report-only` ile aynı mantık).

In [ ]:
results = ev._load_results_file(ev.DEFAULT_RESULTS_FILE)
print(f"Tamamlanan motorlar: {list(results.keys())}\n")
ev._print_summary(results)

## 9. Sunumluk grafik

Motor × (başlık bulma oranı, imza bulma oranı) — gruplu çubuk, tek eksen (%).
İki seri (başlık/imza) sabit renk sırasıyla kodlanmış, değerler doğrudan
etiketlenmiş (renk-tek kodlamadan kaçınmak için) — `dataviz` şablonunun
kategorik iki-seri kuralına uygun.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# dataviz referans paleti (references/palette.md) -- bitişik-çift doğrulanmış
# ilk iki kategorik yuva: mavi (seri 1) / turuncu (seri 2).
COLOR_HEADER = "#2a78d6"
COLOR_SIGNATURE = "#eb6834"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"

engines = list(results.keys())
header_pct, signature_pct = [], []
for name in engines:
    ch = results[name]["totals"]["chain"]
    header_pct.append(100 * ch["header"]["found"] / ch["header"]["expected"])
    signature_pct.append(100 * ch["signature"]["found"] / ch["signature"]["expected"])

x = np.arange(len(engines))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5.5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars1 = ax.bar(x - width / 2, header_pct, width, label="Başlık alanları (Sayı/Tarih/Konu/Muhatap/Gönderen)",
               color=COLOR_HEADER, edgecolor="none")
bars2 = ax.bar(x + width / 2, signature_pct, width, label="İmza alanları (İmza sahibi/unvanı)",
               color=COLOR_SIGNATURE, edgecolor="none")

for bars in (bars1, bars2):
    for b in bars:
        ax.annotate(f"{b.get_height():.0f}%", (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=9, color=TEXT_PRIMARY)

ax.set_ylim(0, 108)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylabel("Alan bulma oranı", color=TEXT_SECONDARY)
ax.set_xticks(x)
ax.set_xticklabels(engines, color=TEXT_PRIMARY)
ax.set_title("Zincir bazında OCR alan kurtarma — gerçek 23 belge / 52 sayfa",
             color=TEXT_PRIMARY, fontsize=13, pad=14)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#d8d7d0")
ax.tick_params(colors=TEXT_SECONDARY, length=0)
ax.grid(axis="y", color="#e8e7e0", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1, frameon=False, fontsize=9, labelcolor=TEXT_SECONDARY)

plt.tight_layout()
plt.savefig("ocr_benchmark.svg", format="svg", bbox_inches="tight")
plt.savefig("ocr_benchmark.png", format="png", dpi=200, bbox_inches="tight")
plt.show()
print("\nKaydedildi: ocr_benchmark.svg, ocr_benchmark.png")

## 10. İndir

Ham JSON'u ve grafiği indirin. JSON'u projeye geri verin --
`docs/evaluation/ocr-benchmark.md` ve `CHANGELOG.md` bu sonuçlarla
yerelde tamamlanacak.

In [ ]:
from google.colab import files
files.download(ev.DEFAULT_RESULTS_FILE)
files.download("ocr_benchmark.svg")
files.download("ocr_benchmark.png")